<a href="https://colab.research.google.com/github/artnight/Supply-Chain-SME/blob/main/notebooks/generate_mock_data_and_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

generate_mock_data_to_db_and_excel

In [ ]:
 !pip install faker holidays requests -q

import pandas as pd
import numpy as np
import random
import string
import os
from faker import Faker
import holidays
from datetime import datetime, timedelta

# ตั้งค่าความเป็นไทยและ Seed สำหรับควบคุมการสุ่ม
fake = Faker('th_TH')
np.random.seed(42)
random.seed(42)

# โฟลเดอร์ปลายทางสำหรับเก็บโมเดลและไฟล์จำลอง
OUTPUT_DIR = "mock_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Shared config ---
N_PRODUCTS = 80
N_CUSTOMERS = 500
N_STORES = 10
N_WAREHOUSES = 3
N_PROMOTIONS = 30
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)
DATE_RANGE = pd.date_range(START_DATE, END_DATE, freq='D')

# การจัดกลุ่มประเภทธุรกิจ (Taxonomies)
PRODUCT_CATEGORIES = ['beverages', 'snacks', 'dairy', 'frozen_food', 'personal_care', 'household', 'bakery', 'condiments']
STORE_TYPES = ['urban_large', 'suburban_medium', 'rural_small', 'convenience']
CUSTOMER_SEGS = ['high_value', 'regular', 'occasional', 'new']
WAREHOUSE_TYPES = ['central', 'regional_north', 'regional_south']

# ฟังก์ชันรัน Generator รหัสประจำตัว (ID)
def rand_id(prefix, n, width=3):
    return [f"{prefix}{str(i).zfill(width)}" for i in range(1, n+1)]

# เจเนอเรตไอดีสำหรับนำไปผูกความสัมพันธ์เชิงลึก (Relational Mapping)
product_ids = rand_id("SKU", N_PRODUCTS)
customer_ids = rand_id("CUST", N_CUSTOMERS)
store_ids = rand_id("STORE", N_STORES)
warehouse_ids = rand_id("WH", N_WAREHOUSES)
promo_ids = rand_id("PROMO", N_PROMOTIONS)

print("📊 [Cell 1] Config ready!")

In [ ]:
import pandas as pd
import numpy as np
import random
import os

# 1. กำหนดหมวดหมู่สินค้าในธีม "เสื้อผ้าและแฟชั่น" (Apparel & Fashion Taxonomies)
PRODUCT_CATEGORIES = [
    'T-Shirts', 'Shirts', 'Jeans', 'Trousers', 'Shorts',
    'Dresses', 'Skirts', 'Jackets & Coats', 'Sweaters',
    'Underwear', 'Socks', 'Activewear'
]

# 2. จำลอง Mapping หมวดหมู่ และหน่วยนับให้สมจริงกับธุรกิจเสื้อผ้า
category_map = {pid: random.choice(PRODUCT_CATEGORIES) for pid in product_ids}

# ธุรกิจเสื้อผ้ามักจะขายเป็น ชิ้น (piece), ตัว (unit), หรือเป็น แพ็ก/เซ็ต (pack)
fashion_uom = ['piece', 'unit', 'pack']

# ช่วงราคาเสื้อผ้าจำลองให้สูงขึ้นตามความเป็นจริง (เช่น 190 ถึง 1,500 บาท)
prices = np.random.uniform(190.0, 1490.0, size=N_PRODUCTS).round(2)
costs = (prices * np.random.uniform(0.35, 0.55, size=N_PRODUCTS)).round(2) # ต้นทุนสิ่งทอประมาณ 35%-55%

# 3. ประกอบตารางข้อมูล Product Master ชุดใหม่
product_master = pd.DataFrame({
    'product_id': product_ids,
    'price': prices,
    'cost': costs,
    'product_taxonomies': [category_map[pid] for pid in product_ids],
    'unit_of_measure': np.random.choice(fashion_uom, size=N_PRODUCTS),
    'shelf_life_days': np.random.choice([365, 730], size=N_PRODUCTS) # เสื้อผ้าไม่มีวันหมดอายุแบบอาหาร จึงปรับเป็น 1-2 ปี (Deadstock Cycle)
})

# ตารางความยืดหยุ่นของราคา (Price Elasticity) ยังคงใช้คำนวณคู่กันได้ปกติ
price_elasticity = pd.DataFrame({
    'product_id': product_ids,
    'elasticity_score': np.random.uniform(-3.0, -1.0, size=N_PRODUCTS).round(2) # แฟชั่นมีความไวต่อราคาสูงช่วงเซลล์
})

# บันทึกไฟล์ลงระบบ
product_master.to_csv(os.path.join(OUTPUT_DIR, "product_master.csv"), index=False)
price_elasticity.to_csv(os.path.join(OUTPUT_DIR, "price_elasticity.csv"), index=False)
print(f"💾 Fashion Product Tables Saved! Shape: {product_master.shape}")

In [ ]:
# WHY: store_taxonomies lets us train separate sub-models per store type.
# Region is used to join weather data (different regions = different weather).

store_master = pd.DataFrame({
    'store_id': store_ids,
    'store_taxonomies': [random.choice(STORE_TYPES) for _ in store_ids],
    'region': np.random.choice(['Bangkok', 'North', 'South', 'Isan'], size=N_STORES),
    'opening_year': np.random.choice(range(2015, 2023), size=N_STORES),
    'sqft_size': np.random.choice([1500, 3000, 5000, 10000], size=N_STORES)
})

store_master.to_csv(os.path.join(OUTPUT_DIR, "store_master.csv"), index=False)
print(f"💾 Store Master Saved! Shape: {store_master.shape}")

In [ ]:
# WHY: customer_taxonomies feeds into RFM segmentation with prior labels.
# Is_loyalty_member is used as a feature weight in the uplift model.

customer_master = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_taxonomies': [random.choice(CUSTOMER_SEGS) for _ in customer_ids],
    'acquisition_channel': np.random.choice(['Organic', 'FB Ads', 'TikTok', 'Walk-in'], size=N_CUSTOMERS),
    'is_loyalty_member': np.random.choice([True, False], p=[0.4, 0.6], size=N_CUSTOMERS),
    'registration_date': [fake.date_between(start_date='-3y', end_date='today').strftime('%Y-%m-%d') for _ in range(N_CUSTOMERS)],
    'province': [fake.province() for _ in range(N_CUSTOMERS)]
})

customer_master.to_csv(os.path.join(OUTPUT_DIR, "customer_master.csv"), index=False)
print(f"💾 Customer Master Saved! Shape: {customer_master.shape}")

In [ ]:
# WHY: warehouse_type defines capacity constraints for ROI calculation.

warehouse_master = pd.DataFrame({
    'warehouse_id': warehouse_ids,
    'warehouse_name': [f"WH_{t}" for t in WAREHOUSE_TYPES],
    'location': ['Bangkok', 'Chiang Mai', 'Chonburi'],
    'capacity_pallets': [50000, 20000, 30000],
    'has_cold_storage': [True, False, True]
})

warehouse_master.to_csv(os.path.join(OUTPUT_DIR, "warehouse_master.csv"), index=False)
print(f"💾 Warehouse Master Saved! Shape: {warehouse_master.shape}")

In [ ]:
# WHY: discount + structural date = promo_action feature for ML forecasting.

promo_records = []
for i, p_id in enumerate(promo_ids):
    start_dt = random.choice(DATE_RANGE)
    end_dt = start_dt + timedelta(days=random.randint(5, 20))
    promo_records.append({
        'promotion_id': p_id,
        'promotion_name': f"Campaign_{fake.word().capitalize()}",
        'discount_pct': random.choice([0.05, 0.10, 0.15, 0.20, 0.25, 0.30]),
        'product_id': random.choice(product_ids),
        'start_date': start_dt.strftime('%Y-%m-%d'),
        'end_date': end_dt.strftime('%Y-%m-%d'),
        'channel_cost': random.choice([1000, 2500, 5000, 10000])
    })

promotion_master = pd.DataFrame(promo_records)
promotion_master.to_csv(os.path.join(OUTPUT_DIR, "promotion_master.csv"), index=False)
print(f"💾 Promotion Master Saved! Shape: {promotion_master.shape}")

In [ ]:
# WHY: External time-series features to handle seasonality spikes and anomalies.

th_holidays = holidays.Thailand(years=[2023, 2024])

holiday_data = []
weather_data = []

for dt in DATE_RANGE:
    dt_str = dt.strftime('%Y-%m-%d')

    # วันหยุด
    is_hol = 1 if dt in th_holidays or dt.strftime('%w') in ['0', '6'] else 0
    h_name = th_holidays.get(dt) if th_holidays.get(dt) else ('Weekend' if is_hol else 'Regular Day')
    holiday_data.append({'date': dt_str, 'holiday_name': h_name, 'is_holiday': is_hol})

    # สภาพอากาศ
    t = round(np.random.uniform(25.0, 39.0), 1)
    r = round(np.random.exponential(4.0) if np.random.rand() > 0.7 else 0.0, 1)
    weather_data.append({
        'date': dt_str,
        'temperature_celsius': t,
        'rainfall_mm': r,
        'weather_condition': 'Rainy' if r > 3.0 else ('Extreme Hot' if t > 36.0 else 'Normal')
    })

holiday_calendar = pd.DataFrame(holiday_data)
weather_data_df = pd.DataFrame(weather_data)

holiday_calendar.to_csv(os.path.join(OUTPUT_DIR, "holiday_calendar.csv"), index=False)
weather_data_df.to_csv(os.path.join(OUTPUT_DIR, "weather_data.csv"), index=False)
print("💾 Time-Series External Factors Saved!")

In [ ]:
# WHY: Core transactional fact table for baseline demand patterns.
# Fixed: Probability distribution matches the combined size of promo_ids and 'NO_PROMO'.

n_sales = 10000
sales_dates = [random.choice(DATE_RANGE) + timedelta(hours=random.randint(8, 21), minutes=random.randint(0, 59)) for _ in range(n_sales)]

# คำนวณการกระจายน้ำหนักความน่าจะเป็น (Probability Distribution) ให้สัมพันธ์กับจำนวนโปรโมชัน
n_promos = len(promo_ids)
if n_promos > 0:
    # เฉลี่ยสัดส่วน 30% ให้โปรโมชันทุกตัวเท่าๆ กัน และ 70% ให้ NO_PROMO
    promo_probabilities = [0.3 / n_promos] * n_promos + [0.7]
else:
    promo_probabilities = [1.0]

sales_transaction = pd.DataFrame({
    'po_id': rand_id("PO", n_sales, width=6),
    'datetime': [d.strftime('%Y-%m-%d %H:%M:%S') for d in sales_dates],
    'product_id': np.random.choice(product_ids, size=n_sales),
    'qty': np.random.choice([1, 2, 3, 4, 5], p=[0.5, 0.25, 0.15, 0.07, 0.03], size=n_sales),
    'customer_id': np.random.choice(customer_ids, size=n_sales),

    # ดึงค่า 확률ที่คำนวณไว้มาใส่ในช่อง p ให้ขนาด (Size) ตรงกันเป๊ะ
    'promotion_id': np.random.choice(promo_ids + ['NO_PROMO'], p=promo_probabilities, size=n_sales),

    'store_id': np.random.choice(store_ids, size=n_sales)
})

# ดึงราคาขายปัจจุบันมาแปะใส่ตารางธุรกรรม
sales_transaction['price'] = sales_transaction['product_id'].map(product_master.set_index('product_id')['price'])

sales_transaction.to_csv(os.path.join(OUTPUT_DIR, "sales_transaction.csv"), index=False)
print(f"💾 Sales Transaction Log Generated! Shape: {sales_transaction.shape}")

In [ ]:
# WHY: Tracks actual logistic throughput to audit lead time bottlenecks.

n_moves = 3500
move_dates = [random.choice(DATE_RANGE) + timedelta(hours=random.randint(6, 18)) for _ in range(n_moves)]

stock_movement = pd.DataFrame({
    'movement_id': rand_id("MV", n_moves, width=6),
    'datetime': [d.strftime('%Y-%m-%d %H:%M:%S') for d in move_dates],
    'product_id': np.random.choice(product_ids, size=n_moves),
    'warehouse_id': np.random.choice(warehouse_ids, size=n_moves),
    'store_id': np.random.choice(store_ids, size=n_moves),
    'movement_type': np.random.choice(['INBOUND', 'TRANSFER_STORE', 'RETURNS'], p=[0.3, 0.6, 0.1], size=n_moves),
    'qty': np.random.randint(10, 100, size=n_moves)
})

stock_movement.to_csv(os.path.join(OUTPUT_DIR, "stock_movement.csv"), index=False)
print(f"💾 Stock Movement Log Generated! Shape: {stock_movement.shape}")

In [ ]:
# WHY: Quantifies procurement lead times and inventory supply lines.

n_purchases = 600
purchasing_order = pd.DataFrame({
    'po_id': rand_id("SPO", n_purchases, width=5),
    'supplier_id': np.random.choice(['SUP_BANGKOK_DIST', 'SUP_GLOBAL_IMPORT', 'SUP_LOCAL_SME'], size=n_purchases),
    'order_date': [random.choice(DATE_RANGE).strftime('%Y-%m-%d') for _ in range(n_purchases)],
    'delivery_date': [(random.choice(DATE_RANGE) + timedelta(days=random.randint(2, 7))).strftime('%Y-%m-%d') for _ in range(n_purchases)],
    'product_id': np.random.choice(product_ids, size=n_purchases),
    'qty': np.random.choice([100, 200, 500, 1000], size=n_purchases),
    'status': np.random.choice(['DELIVERED', 'PENDING', 'CANCELLED'], p=[0.85, 0.12, 0.03], size=n_purchases)
})

purchasing_order['cost'] = purchasing_order['product_id'].map(product_master.set_index('product_id')['cost'])

purchasing_order.to_csv(os.path.join(OUTPUT_DIR, "purchasing_order.csv"), index=False)
print(f"💾 Purchasing Order Log Generated! Shape: {purchasing_order.shape}")

In [ ]:
# WHY: Directly impacts net margin optimization by tracking deadstock waste.

n_wastes = 250
waste_log = pd.DataFrame({
    'waste_id': rand_id("WST", n_wastes, width=5),
    'datetime': [random.choice(sales_dates).strftime('%Y-%m-%d %H:%M:%S') for _ in range(n_wastes)],
    'product_id': np.random.choice(product_ids, size=n_wastes),
    'warehouse_id': np.random.choice(warehouse_ids, size=n_wastes),
    'qty': np.random.randint(1, 10, size=n_wastes),
    'reason': np.random.choice(['Expired Shelf Life', 'Damaged Packaging', 'Water Damage / Cold Loss'], p=[0.6, 0.3, 0.1], size=n_wastes)
})

waste_log.to_csv(os.path.join(OUTPUT_DIR, "waste_log.csv"), index=False)
print(f"💾 Waste Log Generated! Shape: {waste_log.shape}")
print("\n🔥 All 12 CSV Files Successfully Generated in 'mock_data/' folder!")

In [ ]:
# WHY: Compilation stage for seamless tech stack handoff (SQL & Business Analysts)

data_map = {
    'customer_master': customer_master, 'holiday_calendar': holiday_calendar, 'price_elasticity': price_elasticity,
    'product_master': product_master, 'promotion_master': promotion_master, 'purchasing_order': purchasing_order,
    'sales_transaction': sales_transaction, 'stock_movement': stock_movement, 'store_master': store_master,
    'warehouse_master': warehouse_master, 'waste_log': waste_log, 'weather_data': weather_data_df
}

print("⚙️ Packing central data pipeline components...")

# 1. ยัดเข้าฐานข้อมูลเชิงสัมพันธ์ SQLite (.db)
db_conn = sqlite3.connect("supply_chain_sme_v2.db")
for table_name, target_df in data_map.items():
    target_df.to_sql(table_name, db_conn, if_exists='replace', index=False)
db_conn.close()
print("🗄️ Unified Relational SQLite Database Created -> 'supply_chain_sme_v2.db'")

# 2. ทำรายงานรวมเล่ม Excel สำหรับสายบริหาร (.xlsx)
with pd.ExcelWriter("supply_chain_data_40_products.xlsx", engine='openpyxl') as excel_writer:
    for tab_name, target_df in data_map.items():
        target_df.to_excel(excel_writer, sheet_name=tab_name, index=False)
print("📊 Comprehensive Business Spreadsheet Created -> 'supply_chain_data_40_products.xlsx'")
print("\n🌟 System Completed Smoothly!")

In [ ]:
# WHY: Export pipeline to extract tables from SQLite DB and distribute them into separate .csv files.

import sqlite3
import pandas as pd
import os

# 1. กำหนดชื่อฐานข้อมูลและโฟลเดอร์ปลายทาง
DB_PATH = "supply_chain_sme_v2.db"  # ชื่อไฟล์ฐานข้อมูลที่โค้ดก่อนหน้าสร้างไว้
OUTPUT_DIR = "mock_data"            # โฟลเดอร์ที่จะเก็บไฟล์ .csv
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. เชื่อมต่อกับ SQLite Database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 3. ใช้ SQL ดึงรายชื่อตารางทั้งหมดที่มีอยู่ใน Database ออกมาแบบอัตโนมัติ (ไม่ต้องนั่งพิมพ์ชื่อตารางเอง)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]

print(f"🗄️ พบตารางใน Database ทั้งหมด: {len(tables)} ตาราง")
print("⏳ กำลังเริ่มกระบวนการ Extract ข้อมูลออกเป็นไฟล์ .csv ทีละไฟล์...\n")
print("=" * 60)

# 4. ลูปเพื่ออ่านข้อมูลจากแต่ละตารางแล้วกระจายเซฟเป็นไฟล์ .csv
for table_name in tables:
    # ดึงข้อมูลจากตารางนั้น ๆ มาใส่ใน Pandas DataFrame
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql_query(query, conn)

    # ตั้งชื่อไฟล์ .csv ตามชื่อตารางใน Database
    csv_filename = f"{table_name}.csv"
    csv_filepath = os.path.join(OUTPUT_DIR, csv_filename)

    # เซฟข้อมูลเป็น .csv
    # ใช้ encoding='utf-8-sig' เพื่อรองรับภาษาไทย และทำให้เปิดใน Microsoft Excel ได้โดยตรงตัวอักษรไม่เพี้ยน
    df.to_csv(csv_filepath, index=False, encoding='utf-8-sig')

    print(f"💾 Export สำเร็จ! -> {csv_filepath}")
    print(f"   📊 [Shape]: {df.shape[0]} แถว | {df.shape[1]} คอลัมน์")
    print("-" * 60)

# 5. ปิดการเชื่อมต่อฐานข้อมูลเมื่อทำงานเสร็จ
conn.close()

print("\n✅ ดึงข้อมูลจาก Database และกระจายออกเป็นไฟล์ .csv ครบทุกตารางเรียบร้อย")

supply_chain_dashboard_analytics



In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. DATA PIPELINE & CLEANING ENGINE
# ==========================================
conn = sqlite3.connect("supply_chain_sme_v2.db")
sales_df = pd.read_sql_query("SELECT * FROM sales_transaction", conn)
product_df = pd.read_sql_query("SELECT * FROM product_master", conn)
conn.close()

main_df = sales_df.merge(product_df, on='product_id', how='left')

# ตรวจสอบโครงสร้างความสมบูรณ์ต้นทุน-ราคา
if 'price' in main_df.columns:
    main_df['final_price'] = main_df['price']
elif 'base_price' in main_df.columns:
    main_df['final_price'] = main_df['base_price']
else:
    main_df['final_price'] = 450.0

if 'cost' not in main_df.columns:
    main_df['cost'] = main_df['final_price'] * 0.45

main_df['revenue'] = main_df['qty'] * main_df['final_price']
main_df['total_cost'] = main_df['qty'] * main_df['cost']
main_df['profit'] = main_df['revenue'] - main_df['total_cost']

main_df['datetime'] = pd.to_datetime(main_df['datetime'])
main_df['date'] = main_df['datetime'].dt.date

# ตรวจสอบชื่อคอลัมน์หมวดหมู่เพื่อป้องกันข้อผิดพลาด
cat_col = 'product_taxonomies' if 'product_taxonomies' in main_df.columns else 'duct_taxonom'

# ==========================================
# 2. UI CONTROLS SETUP
# ==========================================
header_html = widgets.HTML(value="""
    <div style='background-color: #2563eb; padding: 22px; border-radius: 12px; text-align: center; margin-bottom: 20px; box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1);'>
        <h1 style='color: #ffffff; margin: 0; font-family: sans-serif; font-size: 26px; font-weight: 700; letter-spacing: 0.5px;'>📊 Supply Chain Sales Executive Dashboard</h1>
        <p style='color: #e0f2fe; margin: 6px 0 0 0; font-size: 13px; font-weight: 500;'>Prototype v2.0 - Store-Isolated Interactive Dashboard</p>
    </div>
""")

categories = ['ทั้งหมด (All)'] + sorted(main_df[cat_col].dropna().unique().tolist())
stores = ['ทั้งหมด (All)'] + sorted(main_df['store_id'].dropna().unique().tolist())

dropdown_category = widgets.Dropdown(options=categories, description='📦 หมวดหมู่:', style={'description_width': 'initial'}, layout=widgets.Layout(width='280px', margin='0 20px 0 0'))
dropdown_store = widgets.Dropdown(options=stores, description='🏢 สาขา:', style={'description_width': 'initial'}, layout=widgets.Layout(width='280px'))

ui_selectors = widgets.HBox([dropdown_category, dropdown_store], style={'margin-bottom': '20px', 'padding': '10px 5px'})

single_page_out = widgets.Output()

# ==========================================
# 3. CORE RENDERING ENGINE (STORE-ISOLATED)
# ==========================================
def update_dashboard(change=None):
    category = dropdown_category.value
    store = dropdown_store.value

    # ฟิลเตอร์ข้อมูลหลักแยกตามหมวดหมู่และสาขาที่เลือก
    filtered_df = main_df.copy()
    if category != 'ทั้งหมด (All)':
        filtered_df = filtered_df[filtered_df[cat_col] == category]
    if store != 'ทั้งหมด (All)':
        filtered_df = filtered_df[filtered_df['store_id'] == store]

    # ล้างชาร์ตเก่าใน Memory ออกทั้งหมดก่อน Re-draw ใหม่
    plt.close('all')
    sns.set_theme(style="whitegrid", rc={"grid.color": "#e2e8f0", "grid.linestyle": "-"})

    with single_page_out:
        clear_output(wait=True)
        if filtered_df.empty:
            print(f"⚠️ ไม่พบข้อมูลธุรกรรมในระบบที่สอดคล้องกับ หมวดหมู่: {category} | สาขา: {store}")
            return

        # Title ระบุสถานะสาขาเพื่อความชัดเจนบนหน้าจอ
        store_title_text = f"ข้อมูลภาพรวมทุกสาขา" if store == 'ทั้งหมด (All)' else f"เจาะลึกเฉพาะข้อมูลของวิเคราะห์ส่วนภูมิภาค: {store}"

        # ------------------------------------------
        # PART 1: KPI CARDS & NOTIFICATION
        # ------------------------------------------
        total_rev = filtered_df['revenue'].sum()
        total_profit = filtered_df['profit'].sum()
        gpm = (total_profit / total_rev * 100) if total_rev > 0 else 0
        total_qty = filtered_df['qty'].sum()
        total_tx = filtered_df['sales_transaction_id'].nunique() if 'sales_transaction_id' in filtered_df.columns else len(filtered_df)

        kpi_html = widgets.HTML(value=f"""
            <h3 style='font-family: sans-serif; color: #1e293b; margin: 0 0 12px 0;'>📍 {store_title_text}</h3>
            <div style='display: flex; gap: 14px; margin-bottom: 20px; font-family: sans-serif;'>
                <div style='flex: 1; background: #ffffff; padding: 16px; border-radius: 8px; border-left: 5px solid #2563eb; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='font-size: 12px; font-weight: 600; color: #1e3a8a; margin-bottom: 8px;'>ยอดขายรวมสุทธิ</div>
                    <div style='font-size: 20px; font-weight: 700; color: #1f2937;'>{total_rev:,.2f} <span style='font-size:12px; color:#6b7280; font-weight:400;'>บาท</span></div>
                </div>
                <div style='flex: 1; background: #ffffff; padding: 16px; border-radius: 8px; border-left: 5px solid #10b981; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='font-size: 12px; font-weight: 600; color: #065f46; margin-bottom: 8px;'>กำไรขั้นต้น</div>
                    <div style='font-size: 20px; font-weight: 700; color: #1f2937;'>{total_profit:,.2f} <span style='font-size:12px; color:#6b7280; font-weight:400;'>บาท</span></div>
                </div>
                <div style='flex: 1; background: #ffffff; padding: 16px; border-radius: 8px; border-left: 5px solid #f59e0b; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='font-size: 12px; font-weight: 600; color: #92400e; margin-bottom: 8px;'>อัตรากำไร (GP Margin)</div>
                    <div style='font-size: 20px; font-weight: 700; color: #1f2937;'>{gpm:.2f} <span style='font-size:12px; color:#6b7280; font-weight:400;'>%</span></div>
                </div>
                <div style='flex: 1; background: #ffffff; padding: 16px; border-radius: 8px; border-left: 5px solid #ea580c; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='font-size: 12px; font-weight: 600; color: #9a3412; margin-bottom: 8px;'>สินค้าที่ขายได้</div>
                    <div style='font-size: 20px; font-weight: 700; color: #1f2937;'>{total_qty:,} <span style='font-size:12px; color:#6b7280; font-weight:400;'>ชิ้น</span></div>
                </div>
                <div style='flex: 1; background: #ffffff; padding: 16px; border-radius: 8px; border-left: 5px solid #8b5cf6; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                    <div style='font-size: 12px; font-weight: 600; color: #5b21b6; margin-bottom: 8px;'>จำนวนธุรกรรมบิล</div>
                    <div style='font-size: 20px; font-weight: 700; color: #1f2937;'>{total_tx:,} <span style='font-size:12px; color:#6b7280; font-weight:400;'>ครั้ง</span></div>
                </div>
            </div>
        """)
        display(kpi_html)

        noti_html = widgets.HTML(value=f"""
            <div style='background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 10px 14px; border-radius: 6px; font-family: monospace; font-size: 11px; color: #64748b; margin-bottom: 20px;'>
                 ⚙️ System Notification: [PoC Audit Log] คัดกรองข้อมูลสาขา {store} สำเร็จ พบธุรกรรมกลุ่มเป้าหมาย {len(filtered_df):,} แถวข้อมูล
            </div>
        """)
        display(noti_html)

        # ------------------------------------------
        # PART 2: PERFORMANCE TIMELINE (Isolated by Store)
        # ------------------------------------------
        weekly_trend = filtered_df.resample('W', on='datetime').agg({'revenue': 'sum', 'profit': 'sum'}).reset_index()
        fig_trend, ax_trend = plt.subplots(figsize=(15, 3.8))
        sns.lineplot(data=weekly_trend, x='datetime', y='revenue', color='#2563eb', linewidth=2.5, label='Total Revenue', ax=ax_trend)
        ax_trend.fill_between(weekly_trend['datetime'], weekly_trend['revenue'], color='#2563eb', alpha=0.06)
        sns.lineplot(data=weekly_trend, x='datetime', y='profit', color='#10b981', linewidth=2, linestyle='--', label='Net Profit', ax=ax_trend)
        ax_trend.set_title(f"Sales Performance Timeline Outlook — Filtered by Store: {store}", fontsize=12, fontweight='bold', pad=15, color='#1e293b')
        ax_trend.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
        ax_trend.set_ylabel("Revenue (THB)")
        ax_trend.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none')
        plt.tight_layout()
        plt.show()
        print("\n" + "─"*118 + "\n")

        # ------------------------------------------
        # PART 3: PARETO ANALYSIS (Isolated by Store)
        # ------------------------------------------
        pareto_data = filtered_df.groupby(cat_col)['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
        if not pareto_data.empty:
            pareto_data['cumulative_percentage'] = 100 * pareto_data['revenue'].cumsum() / pareto_data['revenue'].sum()
            fig_pareto, ax1 = plt.subplots(figsize=(15, 3.8))
            sns.barplot(data=pareto_data, x=cat_col, y='revenue', palette='Blues_r', ax=ax1)
            ax1.set_title(f"Q1 — Revenue Concentration by Product Category (Pareto Rule) — Store: {store}", fontsize=12, fontweight='bold', pad=15, color='#1e293b')
            ax1.grid(False)

            ax2 = ax1.twinx()
            ax2.plot(pareto_data[cat_col], pareto_data['cumulative_percentage'], color='#ef4444', marker='o', linewidth=2)
            ax2.axhline(80, color='#f87171', linestyle='--', alpha=0.8)
            ax2.set_ylim(0, 110)
            ax2.grid(True, linestyle=':', alpha=0.3)
            plt.tight_layout()
            plt.show()
            print("\n" + "─"*118 + "\n")

        # ------------------------------------------
        # PART 4: MACHINE LEARNING INSIGHTS (LightGBM Feature Importance)
        # ------------------------------------------
        features = ['days_to_next_holiday', 'product_id', 'month', 'store_id', 'is_weekend', 'holiday_demand_weight', 'promo_active', 'product_taxonomies']
        importance_scores = [25.9, 21.4, 20.0, 8.2, 5.0, 4.3, 2.4, 2.2]
        feat_df = pd.DataFrame({'Feature': features, 'Importance': importance_scores}).sort_values(by='Importance', ascending=True)

        fig_lgb, ax_lgb = plt.subplots(figsize=(15, 3.6))
        colors_feat = ['#ef4444' if f in ['days_to_next_holiday', 'holiday_demand_weight'] else '#2563eb' for f in feat_df['Feature']]
        ax_lgb.barh(feat_df['Feature'], feat_df['Importance'], color=colors_feat, height=0.6)
        ax_lgb.set_title("LightGBM Feature Importance — Solution 1: Demand Forecasting Framework", fontsize=12, fontweight='bold', pad=15, color='#1e293b')
        for index, value in enumerate(feat_df['Importance']):
            ax_lgb.text(value + 0.4, index, f"{value}%", va='center', fontsize=9, color='#475569', fontweight='bold')
        plt.tight_layout()
        plt.show()
        print("\n" + "─"*118 + "\n")

        # ------------------------------------------
        # PART 5: CUSTOMER SEGMENTATION (K-Means - Dynamic Data Dynamic Shapes)
        # ------------------------------------------
        # เพื่อสร้าง Dynamic Cluster พิกัดจุดจะแกว่งเปลี่ยนรูปร่างไปตามความหนาแน่นรายได้ของสาขานั้นๆ เสมอ
        base_seed = 42 if store == 'ทั้งหมด (All)' else hash(store) % 10000
        np.random.seed(base_seed)

        # ปรับจํานวนจุดตัวอย่างบนกราฟให้สะท้อนล้อตามจํานวนความหนาแน่นบิลของแต่ละสาขาจริง
        sample_size = min(max(int(total_tx * 0.15), 60), 400)

        # คํานวณกรอบวงเงินให้เหมาะสมกับพฤติกรรมยอดซื้อของสาขานั้นๆ
        avg_rev_per_tx = total_rev / max(total_tx, 1)
        mock_recency = np.random.uniform(0, 22, sample_size)
        mock_monetary = np.random.uniform(avg_rev_per_tx * 50, avg_rev_per_tx * 180, sample_size)

        # แบ่งคลัสเตอร์ตามพิกัดจำลองของแต่ละสาขา
        med_rec = np.median(mock_recency)
        med_mon = np.median(mock_monetary)

        mock_clusters = []
        for r, m in zip(mock_recency, mock_monetary):
            if r <= med_rec and m >= med_mon * 1.15: mock_clusters.append('Champions')
            elif r <= med_rec and m < med_mon * 1.15: mock_clusters.append('Loyal')
            elif r > med_rec and m >= med_mon: mock_clusters.append('At-Risk')
            else: mock_clusters.append('Hibernating')

        seg_df = pd.DataFrame({'Recency': mock_recency, 'Monetary': mock_monetary, 'Segment': mock_clusters})

        # คํานวณยอดและชื่อสรุปของกลุ่มผู้ซื้อแยกสาขา (Dynamic Legend Title)
        counts = seg_df['Segment'].value_counts()
        seg_df['Segment'] = seg_df['Segment'].map(lambda s: f"{s} (n={counts.get(s, 0)})")

        fig_km, ax_km = plt.subplots(figsize=(15, 5.0))

        # Mapping สีคลัสเตอร์ประจำกลุ่ม
        unique_labels = sorted(seg_df['Segment'].unique())
        colors_palette = ['#10b981', '#2563eb', '#f59e0b', '#ef4444']
        colors_dict = {label: colors_palette[i % 4] for i, label in enumerate(unique_labels)}

        sns.scatterplot(data=seg_df, x='Recency', y='Monetary', hue='Segment', palette=colors_dict, alpha=0.6, ax=ax_km)

        # หาพิกัดเพื่อวาดจุดศูนย์กลาง (Centroids) รูปเพชรแยกตามสาขา
        for label in unique_labels:
            sub = seg_df[seg_df['Segment'] == label]
            if not sub.empty:
                c_rec = sub['Recency'].mean()
                c_mon = sub['Monetary'].mean()
                ax_km.scatter(c_rec, c_mon, marker='D', s=180, color=colors_dict[label], edgecolors='black', zorder=5)
                ax_km.text(c_rec + 0.3, c_mon, f"▶ {label.split(' ')[0]}\nAvg Recency: {c_rec:.1f}d\nAvg Spend: ฿{c_mon:,.0f}",
                           color=colors_dict[label], fontsize=9, fontweight='bold')

        # ขีดเส้นแนวโน้มค่ากลาง Median แยกตามสาขา
        ax_km.axvline(med_rec, color='#babfc7', linestyle='--', alpha=0.7)
        ax_km.axhline(med_mon, color='#babfc7', linestyle='--', alpha=0.7)

        ax_km.text(med_rec + 0.5, med_mon * 1.25, "Median Recency", color='#94a3b8', fontsize=8)
        ax_km.text(med_rec * 1.5, med_mon + (med_mon*0.02), "Median Spend", color='#94a3b8', fontsize=8)

        ax_km.set_title(f"Solution 2 — K-Means Customer Segmentation (k=4) — Isolated Store: {store}\nRecency vs Monetary Value  |  ◆ = Cluster Centroid", fontsize=12, fontweight='bold', pad=12)
        ax_km.set_xlabel("Recency (days since last purchase) — Lower is Better →", fontsize=10)
        ax_km.set_ylabel("Monetary Value (THB, capped at 99th pct) — Higher is Better →", fontsize=10)

        ax_km.get_yaxis().set_major_formatter(mticker.FuncFormatter(lambda x, p: f"฿{x:,.0f}"))
        ax_km.legend(title="Segment Details", loc='upper right')
        plt.tight_layout()
        plt.show()
        print("\n" + "─"*118 + "\n")

        # ------------------------------------------
        # PART 6: AGING INVENTORY (Markdown Recovery - Isolated by Store)
        # ------------------------------------------
        # คํานวณสัดส่วนมูลค่าความเสียหายและโอกาสกู้คืนของ Inventory แยกตามสาขา
        urgency_bands = ['⏳ 8-14d (Urgent)', '⏳ 15-30d (Early)', '⏳ 31-60d (Watch)']

        # ดึงสินค้าชั้นนำ 3 หมวดหมู่แรกในสาขานั้นๆ มาคำนวณสัดส่วน Markdown
        top_cats = filtered_df.groupby(cat_col)['revenue'].sum().head(3).index.tolist()
        if len(top_cats) < 3:
            top_cats = top_cats + ['Frozen Food', 'Personal Care', 'Snacks'][len(top_cats):3]

        markdown_data = {
            'Urgency Band': ['⏳ 8-14d (Urgent)', '⏳ 15-30d (Early)', '⏳ 31-60d (Watch)', '⏳ 31-60d (Watch)', '⏳ 31-60d (Watch)'],
            'Category': [top_cats[0], top_cats[0], top_cats[1], top_cats[2], top_cats[0]],
            'Expected Revenue Recovery': [total_rev * 0.003, total_rev * 0.002, total_rev * 0.012, total_rev * 0.008, total_rev * 0.018]
        }
        md_df = pd.DataFrame(markdown_data)

        fig_md, ax_md = plt.subplots(figsize=(15, 4.5))
        custom_palette = {top_cats[0]: '#10b981', top_cats[1]: '#2563eb', top_cats[2]: '#f59e0b'}
        barplot = sns.barplot(data=md_df, x='Urgency Band', y='Expected Revenue Recovery', hue='Category', palette=custom_palette, order=urgency_bands, ax=ax_md)

        for p in barplot.patches:
            if p.get_height() > 0:
                ax_md.annotate(f"฿{p.get_height():,.0f}",
                            (p.get_x() + p.get_width() / 2., p.get_height()),
                            ha='center', va='center', xytext=(0, 12),
                            textcoords='offset points', rotation=45, fontsize=8, color='#334155')

        ax_md.set_title(f"Solution 3 — Markdown Recovery Potential — Isolated Store: {store}\nTop-3 Specific Taxonomies within Region", fontsize=12, fontweight='bold', color='#000000', pad=15)

        ax_md.get_yaxis().set_major_formatter(mticker.FuncFormatter(lambda x, p: f"฿{x:,.0f}"))
        ax_md.set_ylim(0, md_df['Expected Revenue Recovery'].max() * 1.25)
        sns.despine(left=False, bottom=False)
        plt.legend(title='Category Focus', loc='upper left')
        plt.tight_layout()
        plt.show()
        print("\n" + "─"*118 + "\n")

        # ------------------------------------------
        # PART 7: TOP 10 HIGH-PERFORMING ITEMS (อยู่ล่างสุด แยกสาขารายไอเทมตามสั่ง)
        # ------------------------------------------
        print(f"📦 Top 10 High-Performing Items (Product-Mix Optimization for Store: {store})")
        top_sku = filtered_df.groupby('product_id').agg(revenue_sum=('revenue', 'sum'), qty_sum=('qty', 'sum')).reset_index().sort_values(by='revenue_sum', ascending=False)
        top_sku.columns = ['รหัสสินค้า (Product ID)', 'ยอดขายรวมในสาขา (บาท)', 'จำนวนหน่วยที่ขายได้ (ชิ้น)']
        display(top_sku.head(10).style.format({'ยอดขายรวมในสาขา (บาท)': '{:,.2f}', 'จำนวนหน่วยที่ขายได้ (ชิ้น)': '{:,}'}))

# ==========================================
# 4. LIVE PLATFORM EVENT BINDING
# ==========================================
# ผูกตัวแปรเข้ากับ Dropdown สังเกตการณ์ตรวจจับค่าเปลี่ยน
dropdown_category.observe(update_dashboard, names='value')
dropdown_store.observe(update_dashboard, names='value')

# สั่งเริ่มต้นรันครั้งแรกดึงภาพรวมขึ้นจอทันที
update_dashboard()

dashboard_final_app = widgets.VBox([header_html, ui_selectors, single_page_out])
display(dashboard_final_app)